In [7]:
import pandas as pd
import math

In [8]:
class BestSplit:
  def __init__(self, base_impurity):
    self.base_impurity = base_impurity
    self.feature = None
    self.value = None
    self.is_numeric = None
    self.after_split = None
    self.gain = None
    self.left_labels = None
    self.right_labels = None

  def update(self, feature, value, is_numeric, after_split, gain, left_labels, right_labels):
    self.feature = feature
    self.value = value
    self.is_numeric = is_numeric
    self.after_split = after_split
    self.gain = gain
    self.left_labels = left_labels
    self.right_labels = right_labels

  def has_value(self):
    return self.after_split is not None

  def describe_rule(self):
    if self.is_numeric:
      return f"{self.feature} <= {self.value}"
    else:
      return f"{self.feature} == {self.value}"

  def print_results(self):
    print("Rule:", self.describe_rule())
    print("Before split:", round(self.base_impurity, 4))
    print("After split:", round(self.after_split, 4))
    print("Gain:", round(self.gain, 4))
    print("Left group:", self.left_labels.to_list())
    print("Right group:", self.right_labels.to_list())

In [9]:
def gini_impurity(features):
  n = len(features)
  counts = features.value_counts().tolist()

  gini = 0
  for count in counts:
    p_x = count / n
    gini += p_x * (1 - p_x)

  return gini

def entropy(features):
  n = len(features)
  counts = features.value_counts().tolist()

  entropy = 0
  for count in counts:
    p_x = count / n
    entropy += -p_x * math.log2(p_x)

  return entropy

def weighted_impurity(left_features, right_features, impurity_method):
  left_size = len(left_features)
  right_size = len(right_features)
  total_size = left_size + right_size

  left_impurity = impurity_method(left_features)
  right_impurity = impurity_method(right_features)

  weighted_total = 0
  weighted_total += (left_size / total_size) * left_impurity
  weighted_total += (right_size / total_size) * right_impurity

  return weighted_total

In [10]:
def split_data(df, feature, value, is_numeric):
  if is_numeric:
    left_filter = df[feature] <= value
    right_filter = df[feature] > value
  else:
    left_filter = df[feature] == value
    right_filter = df[feature] != value

  left = df[left_filter]
  right = df[right_filter]
  return left, right

def get_legs_split_values(df):
  # returns the midpoints between the numerical values
  values = sorted(df["Legs"].unique())
  split_values = []

  for i in range(len(values) - 1):
    midpoint = (values[i] + values[i + 1]) / 2
    split_values.append(midpoint)

  return split_values

In [11]:
def update_best_split(best_split, feature, value, is_numeric, after_split, gain, left_labels, right_labels):
  if not best_split.has_value() or after_split < best_split.after_split:
    best_split.update(feature, value, is_numeric, after_split, gain, left_labels, right_labels)

  return best_split


def find_best_split(df, target_column, impurity_function):
  base_impurity = impurity_function(df[target_column])

  best_split = BestSplit(base_impurity)

  # Numeric: Legs
  legs_split_values = get_legs_split_values(df)

  for value in legs_split_values:
    left_group, right_group = split_data(df, "Legs", value, True)

    left_labels = left_group[target_column]
    right_labels = right_group[target_column]

    after_split = weighted_impurity(left_labels, right_labels, impurity_function)
    gain = base_impurity - after_split

    best_split = update_best_split(
      best_split,
      "Legs",
      value,
      True,
      after_split,
      gain,
      left_labels,
      right_labels
    )

  # Categorical: Body Covering
  covering_values = df["Body Covering"].unique()

  for value in covering_values:
    left_group, right_group = split_data(df, "Body Covering", value, False)

    left_labels = left_group[target_column]
    right_labels = right_group[target_column]

    after_split = weighted_impurity(left_labels, right_labels, impurity_function)
    gain = base_impurity - after_split

    best_split = update_best_split(
      best_split,
      "Body Covering",
      value,
      False,
      after_split,
      gain,
      left_labels,
      right_labels
    )

  return best_split

In [12]:
df = pd.read_csv("animals-training.csv")

best_gini_split = find_best_split(df, "Animal", gini_impurity)
best_entropy_split = find_best_split(df, "Animal", entropy)

print("Best Gini Split:")
best_gini_split.print_results()
print()
print("Best Entropy Split:")
best_entropy_split.print_results()


Best Gini Split:
Rule: Legs <= 5.0
Before split: 0.8
After split: 0.5828
Gain: 0.2172
Left group: ['snake', 'snake', 'bird', 'bird', 'bird', 'gorilla', 'gorilla', 'dog', 'dog', 'dog', 'cow']
Right group: ['butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'caterpillar', 'caterpillar']

Best Entropy Split:
Rule: Legs <= 5.0
Before split: 2.5639
After split: 1.5711
Gain: 0.9928
Left group: ['snake', 'snake', 'bird', 'bird', 'bird', 'gorilla', 'gorilla', 'dog', 'dog', 'dog', 'cow']
Right group: ['butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'butterfly', 'caterpillar', 'caterpillar']
